# CD-HIT in Google Colab

Run **CD-HIT** in the browser, with no local installation.

CD-HIT is a widely used program for clustering biological sequences and removing
redundancy. The CD-HIT **web server** has been unavailable since 2022, so this notebook
builds CD-HIT from source in a Colab runtime and runs it on your own data.

**How to use:** run the cells in order. Every parameter is a form field, so the values you
choose are saved inside the notebook — a saved copy is a record of exactly what was run.

---

### Citing

If you use this notebook, please cite **both** CD-HIT and this notebook:

- Li W, Godzik A. *Cd-hit: a fast program for clustering and comparing large sets of
  protein or nucleotide sequences.* Bioinformatics 22:1658-1659 (2006).
- Fu L, Niu B, Zhu Z, Wu S, Li W. *CD-HIT: accelerated for clustering the
  next-generation sequencing data.* Bioinformatics 28:3150-3152 (2012).
- This notebook: see `CITATION.cff` in the repository.

CD-HIT is distributed under the GPL-2.0 licence and is downloaded, not redistributed, by
this notebook. The notebook itself is MIT licensed.


In [ ]:
#@title 1. Install CD-HIT { display-mode: "form" }
#@markdown Builds CD-HIT from source. Takes about a minute on a fresh runtime.

CDHIT_VERSION = "V4.8.1" #@param {type:"string"}

import os, subprocess, shutil, sys, textwrap

WORKDIR = "/content/cdhit_run"
BUILDDIR = "/content/cdhit_build"
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(BUILDDIR, exist_ok=True)

src = f"{BUILDDIR}/cdhit-{CDHIT_VERSION.lstrip('V')}"

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)

if not os.path.exists(f"{src}/cd-hit"):
    print(f"Downloading CD-HIT {CDHIT_VERSION} ...")
    url = f"https://github.com/weizhongli/cdhit/archive/refs/tags/{CDHIT_VERSION}.tar.gz"
    r = sh(f"cd {BUILDDIR} && wget -q {url} -O cdhit.tar.gz && tar -xzf cdhit.tar.gz")
    if r.returncode != 0:
        print(r.stdout); raise SystemExit("Download failed - check the version tag.")

    print("Compiling ...")
    r = sh(f"cd {src} && make -j2")
    if r.returncode != 0:
        print("First build attempt failed; installing build-essential and retrying ...")
        sh("apt-get -qq update && apt-get -qq install -y build-essential")
        r = sh(f"cd {src} && make -j2")
    if r.returncode != 0:
        print(r.stdout[-3000:]); raise SystemExit("Build failed.")

os.environ["PATH"] = src + os.pathsep + os.environ["PATH"]

which = shutil.which("cd-hit")
if not which:
    raise SystemExit("cd-hit is not on PATH after build.")

ver = sh("cd-hit -h | head -3").stdout.strip()
print("cd-hit executable :", which)
print(ver)
print("\nWorking directory :", WORKDIR)


In [ ]:
#@title 2. Provide your FASTA file { display-mode: "form" }
#@markdown Choose where the input comes from. `upload` opens a file picker.
#@markdown `drive` mounts your Google Drive and uses `DRIVE_PATH`.

INPUT_SOURCE = "upload" #@param ["upload", "drive"]
DRIVE_PATH = "/content/drive/MyDrive/my_sequences.fasta" #@param {type:"string"}

import os, shutil, hashlib

INPUT_FASTA = None

if INPUT_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("No file uploaded.")
    name = next(iter(up))
    INPUT_FASTA = os.path.join(WORKDIR, os.path.basename(name))
    with open(INPUT_FASTA, "wb") as fh:
        fh.write(up[name])
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    if not os.path.exists(DRIVE_PATH):
        raise SystemExit(f"Not found: {DRIVE_PATH}")
    INPUT_FASTA = os.path.join(WORKDIR, os.path.basename(DRIVE_PATH))
    shutil.copy(DRIVE_PATH, INPUT_FASTA)

n_seq = 0; lengths = []; cur = 0
with open(INPUT_FASTA) as fh:
    for line in fh:
        if line.startswith(">"):
            n_seq += 1
            if cur: lengths.append(cur)
            cur = 0
        else:
            cur += len(line.strip())
if cur: lengths.append(cur)

INPUT_MD5 = hashlib.md5(open(INPUT_FASTA, "rb").read()).hexdigest()

print("input file :", INPUT_FASTA)
print("sequences  :", n_seq)
if lengths:
    print(f"length     : {min(lengths)}-{max(lengths)} aa/nt, total {sum(lengths)}")
print("md5        :", INPUT_MD5)


In [ ]:
#@title 3. Set parameters and run CD-HIT { display-mode: "form" }
#@markdown **Read the parameter notes carefully** - `-s` and `-aS` mean different things.

#@markdown ---
#@markdown **Sequence identity threshold (`-c`).** Members of a cluster must share at
#@markdown least this fraction of identity with the cluster representative.
IDENTITY = 0.9 #@param {type:"number"}

#@markdown **Word length (`-n`).** Leave at 0 to choose automatically from `-c`
#@markdown (CD-HIT guidance: 5 for 0.7-1.0, 4 for 0.6-0.7, 3 for 0.5-0.6, 2 for 0.4-0.5).
WORD_LENGTH = 0 #@param {type:"integer"}

#@markdown **Minimum length ratio (`-s`).** The *shorter* sequence must be at least this
#@markdown fraction of the representative's **length**. `0.8` means a member must be at
#@markdown least 80% as long as its representative. `0.0` applies no length constraint.
#@markdown This is a minimum, not a maximum difference.
LENGTH_RATIO = 0.0 #@param {type:"number"}

#@markdown **Alignment coverage of the shorter sequence (`-aS`).** The alignment must
#@markdown cover at least this fraction of the shorter sequence. This is usually what is
#@markdown meant by "X% coverage". `0.0` disables it.
COVERAGE_SHORT = 0.0 #@param {type:"number"}

#@markdown **Alignment coverage of the longer sequence (`-aL`).** `0.0` disables it.
COVERAGE_LONG = 0.0 #@param {type:"number"}

#@markdown **Accurate mode (`-g 1`).** Compare each sequence to all representatives and
#@markdown assign it to the most similar, rather than the first match. Slower, recommended.
ACCURATE_MODE = True #@param {type:"boolean"}

#@markdown **Full sequence names in the cluster file (`-d 0`).** Keeps whole FASTA headers
#@markdown instead of truncating at 20 characters. Recommended for traceability.
FULL_NAMES = True #@param {type:"boolean"}

OUTPUT_PREFIX = "cd_hit_output" #@param {type:"string"}

import subprocess, os, datetime

if WORD_LENGTH and WORD_LENGTH > 0:
    n = WORD_LENGTH
else:
    n = 5 if IDENTITY >= 0.7 else 4 if IDENTITY >= 0.6 else 3 if IDENTITY >= 0.5 else 2
    print(f"word length chosen automatically for -c {IDENTITY}: -n {n}")

out = os.path.join(WORKDIR, OUTPUT_PREFIX)

cmd = ["cd-hit", "-i", INPUT_FASTA, "-o", out,
       "-c", str(IDENTITY), "-n", str(n),
       "-g", "1" if ACCURATE_MODE else "0",
       "-d", "0" if FULL_NAMES else "20",
       "-T", "0", "-M", "0"]
if LENGTH_RATIO   > 0: cmd += ["-s",  str(LENGTH_RATIO)]
if COVERAGE_SHORT > 0: cmd += ["-aS", str(COVERAGE_SHORT)]
if COVERAGE_LONG  > 0: cmd += ["-aL", str(COVERAGE_LONG)]

CDHIT_COMMAND = " ".join(cmd)
RUN_TIME = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print("\ncommand:\n ", CDHIT_COMMAND, "\n")
res = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(res.stdout[-4000:])
if res.returncode != 0:
    raise SystemExit(f"CD-HIT failed with exit code {res.returncode}")
print("CD-HIT finished.")


In [ ]:
#@title 4. Summary and provenance record { display-mode: "form" }
#@markdown Summarises the clustering and writes `provenance.txt` next to the results.

import os, re, collections

clstr = out + ".clstr"
sizes = collections.Counter(); reps = []; cur = None
with open(clstr) as fh:
    for line in fh:
        if line.startswith(">Cluster"):
            cur = int(line.split()[1]); sizes[cur] = 0
        elif line.strip():
            sizes[cur] += 1
            if line.rstrip().endswith("*"):
                m = re.search(r">(\S+?)\.\.\.", line) or re.search(r">(\S+)", line)
                reps.append((cur, m.group(1) if m else "?"))

n_clusters = len(sizes)
print(f"{n_seq} sequences -> {n_clusters} clusters")
print("largest clusters:", [c for _, c in sorted(((v, k) for k, v in sizes.items()), reverse=True)[:5]],
      "with", [v for v, _ in sorted(((v, k) for k, v in sizes.items()), reverse=True)[:5]], "members")
print(f"\nrepresentatives written to: {out}")
print(f"cluster assignments:        {clstr}")

prov = f"""CD-HIT run provenance
=====================
date (UTC)      : {RUN_TIME}
CD-HIT version  : {CDHIT_VERSION}
command         : {CDHIT_COMMAND}

input file      : {os.path.basename(INPUT_FASTA)}
input md5       : {INPUT_MD5}
input sequences : {n_seq}

output clusters : {n_clusters}
representatives : {os.path.basename(out)}
cluster file    : {os.path.basename(clstr)}

Notebook: CD-HIT in Google Colab
CD-HIT: Li & Godzik 2006 (Bioinformatics 22:1658-1659);
        Fu et al. 2012 (Bioinformatics 28:3150-3152)
"""
prov_path = os.path.join(WORKDIR, "provenance.txt")
open(prov_path, "w").write(prov)
print("\n" + prov)


In [ ]:
#@title 5. Download the results { display-mode: "form" }
#@markdown Bundles the representatives, the cluster file and the provenance record.

import shutil, os
from google.colab import files

bundle = os.path.join("/content", OUTPUT_PREFIX + "_results")
if os.path.isdir(bundle): shutil.rmtree(bundle)
os.makedirs(bundle)
for f in (out, out + ".clstr", os.path.join(WORKDIR, "provenance.txt")):
    if os.path.exists(f):
        shutil.copy(f, os.path.join(bundle, os.path.basename(f)))
# give the representatives a .fasta extension for convenience
rep = os.path.join(bundle, os.path.basename(out))
if os.path.exists(rep):
    shutil.copy(rep, rep + ".fasta")

archive = shutil.make_archive(bundle, "zip", bundle)
print("bundle:", archive)
files.download(archive)


---

## Parameter notes

| Flag | Meaning |
|---|---|
| `-c` | Sequence identity threshold. Members share at least this identity with the representative. |
| `-n` | Word length used for the initial filter. Must suit `-c`; this notebook can choose it. |
| `-s` | **Minimum length ratio.** The shorter sequence must be at least this fraction of the representative's length. A *minimum*, not a maximum difference. |
| `-aS` | Alignment coverage of the **shorter** sequence. Usually what "% coverage" means. |
| `-aL` | Alignment coverage of the **longer** sequence. |
| `-g 1` | Compare against all representatives and take the best, not the first match. |
| `-d 0` | Keep full sequence names in the `.clstr` file. |

`-s` and `-aS` are easy to confuse. If you want "clustered at 90% identity over at least
80% of the sequence", set `-c 0.9` and `-aS 0.8`.

## Reproducibility

Because every parameter is a Colab form field, saving the notebook after a run preserves
the exact settings used. Cell 4 also writes a `provenance.txt` recording the CD-HIT
version, the full command line, the input filename and its MD5, and the UTC timestamp.
Keep that file with your results.

## Scope and safety

All work happens in `/content/cdhit_run`. The notebook does not modify or delete anything
elsewhere, including mounted Google Drive.
